In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
#
# This notebook is for a simulation of the protocol with an attacker (Eve),
# to demonstrate that the attacker can be detected.
#
# Eve performs an intercept-resend attack:
#   1. She intercepts each qubit Alice sends to Bob.
#   2. She measures it in a randomly chosen basis.
#   3. She re-encodes the measured bit in the same basis she used and
#      forwards it to Bob.
#
# Why this introduces errors:
#   When Eve guesses Alice's basis correctly (prob 1/2), she gets the
#   right bit and Bob's subsequent measurement is undisturbed.
#   When Eve guesses wrong (prob 1/2), the qubit collapses into the
#   wrong basis. Even if Bob then happens to use Alice's correct basis,
#   he now has a 50% chance of getting the wrong bit.
#
#   Result: among positions where Alice and Bob share the same basis,
#   Eve's interception introduces a ~25% error rate.
#   Without any attack the error rate is 0%.
#
# Alice and Bob detect the attack by comparing a public sample of their
# sifted bits. A high error rate (above a threshold) reveals Eve.
#
# Basis convention:
#   basis = 0  ->  standard basis:  |0> = bit 0,  |1> = bit 1
#   basis = 1  ->  diagonal basis:  |+> = bit 0,  |-> = bit 1
#
# All random choices use quantum measurement of |+>.

In [ ]:
# ============================================================
#  SETUP: Simulator and helper functions
# ============================================================

simulator = BasicSimulator()

# Number of qubits Alice sends.
N = 100
SAMPLE_SIZE = 20        # bits sacrificed publicly for eavesdrop detection
DETECTION_THRESHOLD = 0.10  # flag attack if error rate > 10%


def random_bit():
    """
    Generate a single genuinely random bit using quantum measurement.

    Constructs the state |+> = H|0> = (1/sqrt(2))(|0> + |1>) and measures it.
    Each outcome (0 or 1) occurs with probability 1/2 by the Born rule.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)          # |0> -> |+>
    qc.measure(0, 0) # collapse to |0> or |1> with equal probability
    compiled = transpile(qc, simulator)
    result   = simulator.run(compiled, shots=1).result()
    return int(list(result.get_counts().keys())[0])


def encode_qubit(bit, basis):
    """
    Encode a classical bit as a qubit in the chosen basis.

    Standard basis (basis=0):
        bit 0  ->  |0>          (qubits initialise to |0>, do nothing)
        bit 1  ->  |1>          (apply X)

    Diagonal basis (basis=1):
        bit 0  ->  |+> = H|0>   (apply H)
        bit 1  ->  |-> = HX|0>  (apply X then H)

    Returns a QuantumCircuit with the qubit prepared but not yet measured.
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)     # flip |0> to |1>
    if basis == 1:  # diagonal: H maps |0>->|+>, |1>->|->
        qc.h(0)
    return qc


def measure_qubit(qc, basis):
    """
    Measure a qubit (carried in QuantumCircuit qc) in the given basis.

    Standard (basis=0): measure directly.
    Diagonal (basis=1): apply H first, converting |+>->|0> and |->->|1>,
                        then do a standard measurement.

    Returns the measured classical bit (0 or 1).
    """
    if basis == 1:
        qc.h(0)      # rotate diagonal basis into standard basis
    qc.measure(0, 0)
    compiled = transpile(qc, simulator)
    result   = simulator.run(compiled, shots=1).result()
    return int(list(result.get_counts().keys())[0])


def basis_name(b):
    """Return a readable label for basis index b."""
    return 'std' if b == 0 else 'diag'

In [ ]:
# ============================================================
#  ALICE
#  Step 1: Generate a random bit string and random basis choices.
#  Step 2: Encode each bit as a qubit and send it -- but Eve
#           intercepts every qubit before it reaches Bob.
# ============================================================

print('=== ALICE ===')
print(f'Preparing {N} qubits to send to Bob.')
print('All random choices made by measuring |+> (genuine quantum randomness).\n')

alice_bits  = [random_bit() for _ in range(N)]
alice_bases = [random_bit() for _ in range(N)]   # 0=standard, 1=diagonal

# Encode each bit as a qubit circuit
alice_qubits = []
for i in range(N):
    qc = encode_qubit(alice_bits[i], alice_bases[i])
    alice_qubits.append(qc)

print(f'Alice bits  (first 20): {alice_bits[:20]}')
print(f'Alice bases (first 20): {[basis_name(b) for b in alice_bases[:20]]}')
print('\nAlice has sent her qubits -- but Eve intercepts them in transit!')

In [ ]:
# ============================================================
#  EVE (ATTACKER) -- intercept-resend attack
#
#  Eve intercepts every qubit on the Alice -> Bob channel.
#  For each qubit she:
#    1. Randomly picks a basis to measure in (quantum random choice).
#    2. Measures the qubit, collapsing its state.
#    3. Re-encodes her measured bit in the same basis she chose.
#    4. Forwards the new (possibly disturbed) qubit to Bob.
#
#  Eve cannot copy the qubit (No-Cloning Theorem), so she must
#  destroy the original to get any information from it.
#
#  Error analysis for positions where Alice and Bob share a basis:
#    Case 1 -- Eve's basis == Alice's basis (prob 1/2):
#      Eve measures correctly, gets the right bit, re-encodes correctly.
#      Bob's result is undisturbed.  Error probability: 0
#
#    Case 2 -- Eve's basis != Alice's basis (prob 1/2):
#      Eve's measurement collapses the qubit into her (wrong) basis.
#      She re-encodes her random result in the wrong basis.
#      Bob measures in Alice's (correct) basis from Eve's qubit,
#      getting the right answer only 50% of the time.
#      Error probability: 1/2
#
#  Overall error rate (on matched positions) = 1/2 * 1/2 = 25%
# ============================================================

print('=== EVE (ATTACKER) ===')
print('Eve intercepts every qubit, measures it, then re-encodes and forwards.\n')

eve_bases        = [random_bit() for _ in range(N)]  # Eve's random basis choices
eve_results      = []   # bits Eve measured (her partial view of the key)
forwarded_qubits = []   # qubits Eve sends on to Bob

for i in range(N):
    # --- Eve intercepts and measures Alice's qubit ---
    qc_intercept = QuantumCircuit(1, 1)
    qc_intercept.compose(alice_qubits[i], inplace=True)   # Alice's encoding
    eve_bit = measure_qubit(qc_intercept, eve_bases[i])    # Eve measures
    eve_results.append(eve_bit)

    # --- Eve re-encodes her measured bit and forwards to Bob ---
    # She uses the same basis she measured in, which may differ from Alice's.
    qc_forward = encode_qubit(eve_bit, eve_bases[i])
    forwarded_qubits.append(qc_forward)

print(f'Eve bases   (first 20): {[basis_name(b) for b in eve_bases[:20]]}')
print(f'Eve results (first 20): {eve_results[:20]}')

# Count how often Eve chose the same basis as Alice (she can't know this)
eve_correct_bases = sum(eve_bases[i] == alice_bases[i] for i in range(N))
print(f'\nEve chose Alice\'s basis {eve_correct_bases}/{N} times '
      f'({100*eve_correct_bases/N:.1f}% -- expected ~50%).')
print('Eve has forwarded (disturbed) qubits to Bob.')

In [ ]:
# ============================================================
#  EXTENSION: PARTIAL INTERCEPT ATTACK (SIMULATED)
#
#  Eve intercepts each qubit independently with probability ~50%,
#  decided by a quantum coin flip (random_bit()). Intercepted qubits
#  are measured and re-encoded (same as the full attack). Unintercepted
#  qubits are passed through to Bob unchanged.
#
#  Expected error rate on sifted bits:
#    = P(intercept) * P(Eve wrong basis) * P(Bob wrong bit | Eve wrong)
#    = 0.5 * 0.5 * 0.5 = 12.5%   (vs ~25% for full intercept)
#
#  A lower error rate means harder to detect -- but Eve also gains
#  less information about the key.
# ============================================================

print('=== EXTENSION: PARTIAL INTERCEPT ATTACK ===')
print('Eve intercepts each qubit with probability ~50% (quantum coin flip).\n')

partial_eve_bases  = [random_bit() for _ in range(N)]
partial_intercepts = [random_bit() for _ in range(N)]  # 0=intercept, 1=pass through
partial_forwarded  = []
n_intercepted      = 0

for i in range(N):
    if partial_intercepts[i] == 0:       # Eve intercepts this qubit
        qc_int = QuantumCircuit(1, 1)
        qc_int.compose(alice_qubits[i], inplace=True)
        eve_bit = measure_qubit(qc_int, partial_eve_bases[i])
        qc_fwd  = encode_qubit(eve_bit, partial_eve_bases[i])
        n_intercepted += 1
    else:                                # Eve passes this qubit unchanged
        qc_fwd = QuantumCircuit(1, 1)
        qc_fwd.compose(alice_qubits[i], inplace=True)
    partial_forwarded.append(qc_fwd)

print(f'Qubits intercepted: {n_intercepted}/{N} ({100*n_intercepted/N:.1f}% -- expected ~50%)')

# Bob measures the partially-forwarded qubits
partial_bob_bases   = [random_bit() for _ in range(N)]
partial_bob_results = []
for i in range(N):
    qc = QuantumCircuit(1, 1)
    qc.compose(partial_forwarded[i], inplace=True)
    partial_bob_results.append(measure_qubit(qc, partial_bob_bases[i]))

# Sift
p_matching     = [i for i in range(N) if alice_bases[i] == partial_bob_bases[i]]
p_alice_sifted = [alice_bits[i]          for i in p_matching]
p_bob_sifted   = [partial_bob_results[i] for i in p_matching]
p_errors       = sum(a != b for a, b in zip(p_alice_sifted, p_bob_sifted))
p_error_rate   = p_errors / len(p_alice_sifted) if p_alice_sifted else 0

print(f'Sifted bits:  {len(p_alice_sifted)}')
print(f'Errors:       {p_errors}/{len(p_alice_sifted)} = {p_error_rate:.2%}')
print(f'Expected:     ~12.5%  (full intercept -> ~25%)')
print(f'Threshold:    {DETECTION_THRESHOLD:.0%}')
print()
if p_error_rate > DETECTION_THRESHOLD:
    print(f'=> DETECTABLE ({p_error_rate:.2%} > {DETECTION_THRESHOLD:.0%})')
else:
    print(f'=> BELOW threshold ({p_error_rate:.2%} <= {DETECTION_THRESHOLD:.0%}) -- Eve may evade detection')
print(f'P(Eve undetected | sample={SAMPLE_SIZE}): '
      f'{((1 - p_error_rate) ** SAMPLE_SIZE) * 100:.2f}%')

In [ ]:
# ============================================================
#  BOB
#  Bob receives the qubits -- but they have been tampered with by Eve.
#  He measures each in a randomly chosen basis, as in the plain protocol.
# ============================================================

print('=== BOB ===')
print('Bob receives the (forwarded) qubits and measures each in a random basis.\n')

bob_bases   = [random_bit() for _ in range(N)]
bob_results = []

for i in range(N):
    qc = QuantumCircuit(1, 1)
    qc.compose(forwarded_qubits[i], inplace=True)  # Eve's re-encoded qubit
    bit = measure_qubit(qc, bob_bases[i])
    bob_results.append(bit)

print(f'Bob bases   (first 20): {[basis_name(b) for b in bob_bases[:20]]}')
print(f'Bob results (first 20): {bob_results[:20]}')
print('\nBob has measured all qubits. His results are private for now.')

In [ ]:
# ============================================================
#  PUBLIC CHANNEL: BASIS RECONCILIATION (SIFTING)
#  Alice and Bob compare their basis choices publicly.
#  Only positions where both used the same basis are kept.
#  (Eve learns which positions will form the key -- but she
#   has already been detected in the next step.)
# ============================================================

print('=== PUBLIC CHANNEL: BASIS RECONCILIATION ===')
print('Bob announces his basis choices. Alice replies with which ones match.\n')

matching_positions = [i for i in range(N) if alice_bases[i] == bob_bases[i]]

alice_sifted = [alice_bits[i]  for i in matching_positions]
bob_sifted   = [bob_results[i] for i in matching_positions]

print(f'Total qubits sent:        {N}')
print(f'Matching basis positions: {len(matching_positions)} '
      f'({100*len(matching_positions)/N:.1f}%)')
print(f'\nSifted key length: {len(alice_sifted)} bits')
print(f'Alice sifted (first 20): {alice_sifted[:20]}')
print(f'Bob   sifted (first 20): {bob_sifted[:20]}')

# Note: we can peek at the true error for analysis (not available in real QKD)
true_errors = sum(a != b for a, b in zip(alice_sifted, bob_sifted))
true_error_rate = true_errors / len(alice_sifted) if alice_sifted else 0
print(f'\n[Analysis] True error rate over ALL sifted bits: '
      f'{true_errors}/{len(alice_sifted)} = {true_error_rate:.2%} '
      f'(theoretical expectation ~25%)')

In [ ]:
# ============================================================
#  PUBLIC CHANNEL: EAVESDROP DETECTION
#
#  Alice and Bob publicly compare a RANDOM sample of SAMPLE_SIZE sifted bits.
#  Sample indices are chosen using quantum randomness (random_bit()).
#  Without an attacker: error rate ~0%.
#  With Eve's full intercept-resend attack: error rate ~25%.
#
#  If error_rate > DETECTION_THRESHOLD, the protocol aborts.
#
#  The probability that Eve goes UNDETECTED with a sample of k bits:
#    P(undetected) = (0.75)^k
#    k=10 -> 5.6%,  k=20 -> 0.32%,  k=30 -> 0.018%
# ============================================================

print('=== EAVESDROP DETECTION ===')
print(f'Alice and Bob publicly compare a random sample of {SAMPLE_SIZE} sifted bits.')
print(f'Detection threshold: error rate > {DETECTION_THRESHOLD:.0%}\n')

# Select SAMPLE_SIZE unique indices using quantum randomness (rejection sampling).
seen = set()
sample_indices = []
while len(sample_indices) < SAMPLE_SIZE:
    bits_needed = max(1, (len(alice_sifted) - 1).bit_length())
    idx = sum(random_bit() << k for k in range(bits_needed)) % len(alice_sifted)
    if idx not in seen:
        seen.add(idx)
        sample_indices.append(idx)
sample_indices.sort()

sample_alice = [alice_sifted[i] for i in sample_indices]
sample_bob   = [bob_sifted[i]   for i in sample_indices]

errors     = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate = errors / SAMPLE_SIZE

print(f'Sample (Alice): {sample_alice}')
print(f'Sample (Bob):   {sample_bob}')
print(f'\nErrors in sample:  {errors} / {SAMPLE_SIZE}')
print(f'Error rate:        {error_rate:.2%}')

p_undetected = (0.75 ** SAMPLE_SIZE) * 100
print(f'Theoretical P(Eve undetected with {SAMPLE_SIZE} bits): {p_undetected:.4f}%\n')

print('-' * 55)
if error_rate > DETECTION_THRESHOLD:
    print('*** ATTACK DETECTED! ***')
    print(f'Error rate {error_rate:.2%} exceeds the threshold of {DETECTION_THRESHOLD:.0%}.')
    print('Alice and Bob abort the key exchange and try again')
    print('on a channel known to be free of interference.')
    print('-' * 55)
    print(f'\n[Analysis] Eve measured the correct basis '
          f'{sum(eve_bases[i]==alice_bases[i] for i in matching_positions)}/'
          f'{len(matching_positions)} times on sifted positions ({"~50% as expected"}).')
    print('Eve\'s partial key guess (first 20):', eve_results[:20])
    print('Eve\'s guesses on sifted bits (first 20):',
          [eve_results[i] for i in matching_positions[:20]])
    print('True Alice sifted bits (first 20):   ', alice_sifted[:20])
else:
    key_indices = [i for i in range(len(alice_sifted)) if i not in set(sample_indices)]
    final_key   = [alice_sifted[i] for i in key_indices]
    print(f'Attack not detected in this sample (Eve was lucky -- probability {p_undetected:.4f}%).')
    print('In practice, increase SAMPLE_SIZE to make this risk negligible.')
    print(f'Key established ({len(final_key)} bits), but it may be compromised.')